# Conversation Cleaning

Merges consecutive same-speaker rows in the raw CSV into single logical turns.

**Why:** WeChat lets you send multiple messages in a burst — each message is its own CSV row.
The model and ORPO pipeline expect one merged reply per turn, not fragments.

**Approach:** Deterministic rules — group consecutive rows with the same `speaker` within
each conversation and join their text with `\n`.

In [12]:
from pathlib import Path

try:
    _ROOT = Path(__vsc_ipynb_file__).parent.parent  # chatbot/
except NameError:
    _ROOT = Path.cwd()

IN_EXCEL = "/Users/school/Downloads/all_messages(1).xlsx"
OUT_JSON = str(_ROOT / "output" / "all_messages_cleaned.json")
OUT_CSV  = str(_ROOT / "output" / "all_messages_cleaned.csv")

print("Project root:", _ROOT)
print("IN_EXCEL :", IN_EXCEL)

Project root: /Users/school/Ian's Chatbot/chatbot
IN_EXCEL : /Users/school/Downloads/all_messages(1).xlsx


In [13]:
import pandas as pd
import json, re

df = pd.read_excel(IN_EXCEL,header=1, dtype=str)
print(f"Loaded {len(df):,} rows across {df['conversation_id'].nunique()} conversations")
print("Columns:", df.columns.tolist())
df.head(10)

Loaded 2,620 rows across 73 conversations
Columns: ['row_id', 'conversation_id', 'speaker', 'Unnamed: 2', 'text', 'confidence', 'y_center', 'mean_h', 'mean_s', 'mean_v', 'source_image', 'ocr_status', 'edited', 'locked']


,row_id,conversation_id,speaker,Unnamed: 2,text,confidence,y_center,mean_h,mean_s,mean_v,source_image,ocr_status,edited,locked
0,1c1d2dcd4f34df0beb88376fb7c66210ba4e5fd7,convo1,user,NaN,呐,0.99175500869751,503,50.4043209876543,143.889814814815,209.243364197531,IMG_7788.jpg,ok,1,1
1,4dd77af18e4972a6c93f59bc09470c2a49cf7b43,convo1,user,NaN,把好天气分享给你,0.998297095298767,670,50.4838836220987,145.671624713959,200.888166067342,IMG_7788.jpg,ok,1,1
2,7c99e2c3caf069f06751fed470c23a78783e980c,convo1,assistant,NaN,哇哇哇,0.994340598583221,835,0,0,217.352745183254,IMG_7788.jpg,ok,1,1
3,15ce930e281324e5fecd07f6b33bc169ec3f6237,convo1,assistant,NaN,是谁有这么好的运气能被分享好天气！,0.997438549995422,1003,0,0,220.218136413298,IMG_7788.jpg,ok,1,1
4,201fa9d24574ebb505d8fcda5d10f1b5883fcb7b,convo1,assistant,NaN,NaN,0.987993180751801,1059.5,0,0,232.455615060045,IMG_7788.jpg,ok,1,1
5,4e18fa7bb219dc06f68976dac81f8d6c30b6a25f,convo1,assistant,NaN,是谁！,0.980450868606567,1216.25,0,0,227.6090805718,IMG_7788.jpg,ok,1,1
6,ae9846107129e1b3ff149537678c0b93f1d8fc40,convo1,assistant,NaN,原来是我这个幸运的人,0.99760514497757,1378,0,0,230.286776061776,IMG_7788.jpg,ok,1,1
7,97b2744442c2cd869a889b6f9e07bd83bd7a2865,convo11,user,NaN,被采购气死了,0.996510684490204,34.5,47.9028944911298,126.925848117025,204.593915343915,IMG_9228.jpg,ok,1,1
8,161ca84d9345d18dff8174e8ed3680eaed49f3ea,convo11,user,NaN,就是上次和你说的那个,0.997275531291962,136,47.8279258886653,126.14067739772,206.366616364856,IMG_9228.jpg,ok,1,1
9,ad59501a2aa5a2bac82a905671bbd4535179a82e,convo11,user,NaN,无语,0.999482870101929,245,47.729475308642,125.730864197531,211.418672839506,IMG_9228.jpg,ok,1,1


In [14]:
# ── Sort ──────────────────────────────────────────────────────────────────────
def image_number(fname: str) -> int:
    m = re.search(r'(\d+)', str(fname))
    return int(m.group(1)) if m else 0

df['image_number'] = df['source_image'].apply(image_number)
df['y_center'] = pd.to_numeric(df['y_center'], errors='coerce').fillna(0).astype(int)
df = df.sort_values(['conversation_id', 'image_number', 'y_center']).reset_index(drop=True)
print("Sorted by (conversation_id, image_number, y_center)")
df[df['conversation_id'] == 'convo1'][['conversation_id', 'source_image', 'image_number', 'y_center', 'speaker', 'text']].head(20)

Sorted by (conversation_id, image_number, y_center)


,conversation_id,source_image,image_number,y_center,speaker,text
0,convo1,IMG_7788.jpg,7788,503,user,呐
1,convo1,IMG_7788.jpg,7788,670,user,把好天气分享给你
2,convo1,IMG_7788.jpg,7788,835,assistant,哇哇哇
3,convo1,IMG_7788.jpg,7788,1003,assistant,是谁有这么好的运气能被分享好天气！
4,convo1,IMG_7788.jpg,7788,1059,assistant,NaN
5,convo1,IMG_7788.jpg,7788,1216,assistant,是谁！
6,convo1,IMG_7788.jpg,7788,1378,assistant,原来是我这个幸运的人


In [15]:
# ── Merge consecutive same-speaker rows (within each conversation + source_image) ──
merged_rows = []   # list of dicts for the cleaned CSV
merge_log   = []   # for the sample review cell

def is_blank(val) -> bool:
    """True if the cell is any kind of pandas/Python NA or the literal string 'nan'."""
    try:
        if pd.isna(val):
            return True
    except (TypeError, ValueError):
        pass
    return str(val).strip().lower() == "nan"

# Group by (conversation_id, source_image) — merge only within the same screenshot
for (conv_id, src_img), group in df.groupby(['conversation_id', 'source_image'], sort=False):
    group = group.reset_index(drop=True)
    i = 0
    while i < len(group):
        current_speaker = group.at[i, 'speaker']
        fragments = []
        j = i
        # Collect consecutive rows with the same speaker in this screenshot
        while j < len(group) and group.at[j, 'speaker'] == current_speaker:
            row = group.iloc[j]
            if not is_blank(row['text']):
                fragments.append(row)
            j += 1

        if fragments:
            merged_text = '\n'.join(str(r['text']) for r in fragments)
            merged_locked = int(all(int(r.get('locked', 0)) == 1 for r in fragments))

            if len(fragments) > 1:
                merge_log.append({
                    'conversation_id': conv_id,
                    'source_image':    src_img,
                    'speaker':         current_speaker,
                    'fragments':       [str(r['text']) for r in fragments],
                    'merged':          merged_text,
                })

            last = fragments[-1]
            merged_rows.append({
                'conversation_id': conv_id,
                'speaker':         current_speaker,
                'text':            merged_text,
                'y_center':        last['y_center'],
                'source_image':    src_img,
                'image_number':    image_number(src_img),
                'locked':          merged_locked,
                'fragment_count':  len(fragments),
            })
        i = j

clean_df = pd.DataFrame(merged_rows)

# Re-sort to guarantee correct chronological order for JSON/CSV export
clean_df = clean_df.sort_values(['conversation_id', 'image_number', 'y_center']).reset_index(drop=True)

print(f"Before: {len(df):,} rows → After: {len(clean_df):,} rows  ({len(df)-len(clean_df):,} rows merged/dropped)")
print(f"Total merge events (2+ fragments): {len(merge_log)}")

Before: 2,620 rows → After: 732 rows  (1,888 rows merged/dropped)
Total merge events (2+ fragments): 379


In [18]:
clean_df.head(20)

,conversation_id,speaker,text,y_center,source_image,image_number,locked,fragment_count
0,convo1,user,呐\n把好天气分享给你,670,IMG_7788.jpg,7788,1,2
1,convo1,assistant,哇哇哇\n是谁有这么好的运气能被分享好天气！\n是谁！\n原来是我这个幸运的人,1378,IMG_7788.jpg,7788,1,4
2,convo11,user,被采购气死了\n就是上次和你说的那个\n无语\n沟通太费劲了\n厌蠢症犯了,477,IMG_9228.jpg,9228,1,5
3,convo11,assistant,又犯蠢了？\n你直接开骂\n不要忍着\n给你点杯苦瓜柠檬茶消消火?,1154,IMG_9228.jpg,9228,1,4
4,convo11,user,我已經很苦了,10,IMG_9229.jpg,9229,1,1
5,convo11,assistant,那喝杯其他的？\n其实不苦 柠檬味道为主\n我喝过,309,IMG_9229.jpg,9229,1,3
6,convo11,user,算了不让你破费了\n哈哈哈哈,556,IMG_9229.jpg,9229,1,2
7,convo11,assistant,笑死\n主要是不让我破费是吧,982,IMG_9229.jpg,9229,1,2
8,convo11,user,哈哈哈哈哈,1100,IMG_9229.jpg,9229,1,1
9,convo11,assistant,这不叫破费，给你买开心,1195,IMG_9229.jpg,9229,1,1


In [ ]:
# # ── Export merge log for manual inspection ────────────────────────────────────
# merge_log_path = str(_ROOT / "output" / "merge_review.csv")

# merge_log_df = pd.DataFrame([
#     {
#         'conversation_id': m['conversation_id'],
#         'source_image':    m['source_image'],
#         'speaker':         m['speaker'],
#         'fragment_count':  len(m['fragments']),
#         'fragments':       ' | '.join(m['fragments']),
#         'merged':          m['merged'],
#     }
#     for m in merge_log
# ])

# merge_log_df.to_csv(merge_log_path, index=False)
# print(f"Wrote {len(merge_log_df)} merge events to {merge_log_path}")
# print("Open output/merge_review.csv to inspect before running the export cells below.")

Wrote 379 merge events to /Users/school/Ian's Chatbot/chatbot/output/merge_review.csv
Open output/merge_review.csv to inspect before running the export cells below.


In [20]:
# ── Export JSON ───────────────────────────────────────────────────────────────
# Format: {conv_id: [{"role": "user"|"assistant", "content": "...", "locked": 0|1}, ...]}

ROLE_MAP = {'user': 'user', 'assistant': 'assistant'}

conversations = {}
for _, row in clean_df.iterrows():
    cid = row['conversation_id']
    if cid not in conversations:
        conversations[cid] = []
    conversations[cid].append({
        'role':    ROLE_MAP.get(row['speaker'], row['speaker']),
        'content': row['text'],
        'locked':  int(row['locked']),
    })

with open(OUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(conversations, f, ensure_ascii=False, indent=2)

print(f"Wrote {len(conversations)} conversations to {OUT_JSON}")

Wrote 73 conversations to /Users/school/Ian's Chatbot/chatbot/output/all_messages_cleaned.json


In [21]:
# ── Export cleaned CSV ────────────────────────────────────────────────────────
clean_df.to_csv(OUT_CSV, index=False)
print(f"Wrote {len(clean_df):,} rows to {OUT_CSV}")

Wrote 732 rows to /Users/school/Ian's Chatbot/chatbot/output/all_messages_cleaned.csv
